# Step 4 — Map & traffic validation (safe first run)


In [13]:
import sys
from IPython.display import display
from routing.config import ORIGIN, DESTINATION
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph, preview_probes, save_summary
print('Python:', sys.executable)

Python: /Users/mojtabashabani/wiser/mary/.venv/bin/python


In [14]:

RADIUS_M = 1200
G = load_road_graph(radius_m=RADIUS_M)
print('Graph:', type(G).__name__, G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')
assert G.is_directed()
assert all(float(d['length']) >= 0 for *_, d in G.edges(keys=True, data=True))
display(preview_probes(G, limit=5))

Graph: MultiDiGraph 747 nodes, 1718 edges


[{'edge_id': (25291709, 3764277424, 0),
  'latitude': 51.4999758,
  'longitude': -0.1753925,
  'length_m': 19.8},
 {'edge_id': (1528999913, 198021, 0),
  'latitude': 51.4887435,
  'longitude': -0.1927866,
  'length_m': 44.0},
 {'edge_id': (37873014, 37878510, 0),
  'latitude': 51.4889177,
  'longitude': -0.1606918,
  'length_m': 27.2},
 {'edge_id': (102094, 102098, 0),
  'latitude': 51.5059661,
  'longitude': -0.1922284,
  'length_m': 302.5},
 {'edge_id': (5154417295, 26389732, 0),
  'latitude': 51.5003095,
  'longitude': -0.1588334,
  'length_m': 13.6}]

In [15]:

basic_widget, basic_state = interactive_map(G, origin=ORIGIN, destination=DESTINATION)
display(basic_widget)
print('Double-click on the map, then run the NEXT cell.')

Double-click on the map, then run the NEXT cell.


In [16]:
print('Origin node:', basic_state['origin_node'])
print('Destination node:', basic_state['destination_node'])
print('Last click:', basic_state['last_click_latlon'])
if basic_state['last_click_latlon'] is None:
    print('No double-click yet; map must render and receive an interaction.')
else:
    assert basic_state['origin_node'] in G
    assert basic_state['destination_node'] in G
    print('PASS — interactive origin')

Origin node: 25291778
Destination node: 26389375
Last click: None
No double-click yet; map must render and receive an interaction.


In [26]:

RUN_LIVE_TRAFFIC = True
REQUEST_LIMIT = 12

if RUN_LIVE_TRAFFIC:
    with TomTomClient() as client:
        snapshot, summary = collect_traffic_snapshot(
            G, client, max_requests=REQUEST_LIMIT,
        )
    save_summary(summary)  
    print('LIVE TomTom snapshot collected')
else:

    snapshot, summary = collect_traffic_snapshot(G, client=None, max_requests=0)
    print('OFFLINE fallback-only snapshot: ZERO TomTom requests')

print('Requests:', summary['requests_made'])
print('Usable coverage (%):', summary['usable_coverage_percent'])
print('Usable among sampled (%):', summary['usable_among_sampled_percent'])
print('Counts:', summary['counts'])
print('Mapping diagnostics:', summary['map_matching_diagnostics'])
print('API error groups:', summary['api_error_groups'])

LIVE TomTom snapshot collected
Requests: 12
Usable coverage (%): 0.09
Usable among sampled (%): 41.67
Counts: {'missing': 5845, 'red': 0, 'yellow': 1, 'clear': 3, 'closed': 1}
Mapping diagnostics: {'candidate_match': 5, 'heading_proxy_mismatch': 2, 'midpoint_too_far': 4}
API error groups: [{'http_status': 400, 'error_code': 'INVALID_REQUEST', 'count': 1, 'example_message_redacted': 'Point too far from nearest existing segment.'}]


In [28]:
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot

with TomTomClient() as client:
    snapshot, summary = collect_traffic_snapshot(
        G,
        client,
        max_requests=REQUEST_LIMIT
    )

print("Requests:", summary["requests_made"])
print("Errors:", summary["request_errors"])
print("Coverage:", summary["usable_coverage_percent"])

print("Matching diagnostics:")
print(summary["map_matching_diagnostics"])

print("API errors:")
print(summary["api_error_groups"])

Requests: 12
Errors: 1
Coverage: 0.09
Matching diagnostics:
{'candidate_match': 5, 'heading_proxy_mismatch': 2, 'midpoint_too_far': 4}
API errors:
[{'http_status': 400, 'error_code': 'INVALID_REQUEST', 'count': 1, 'example_message_redacted': 'Point too far from nearest existing segment.'}]


In [29]:
traffic_widget, traffic_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot
)

display(traffic_widget)

In [19]:
free_time_graph = make_routing_graph(snapshot, objective='time', scenario='free_flow')
measured_time_graph = make_routing_graph(snapshot, objective='time', scenario='measured')
measured_distance_graph = make_routing_graph(snapshot, objective='distance', scenario='measured')
print('Free-flow time edges:', free_time_graph.number_of_edges())
print('Measured time edges:', measured_time_graph.number_of_edges())
print('Measured distance edges:', measured_distance_graph.number_of_edges())
assert all(float(d['length']) >= 0 for *_, d in measured_time_graph.edges(keys=True, data=True))
print('PASS — routing graphs prepared; no search algorithm executed')

Free-flow time edges: 1718
Measured time edges: 1718
Measured distance edges: 1718
PASS — routing graphs prepared; no search algorithm executed


In [20]:
from routing.map_loader import load_road_graph

RADIUS_M = 2000

G = load_road_graph(radius_m=RADIUS_M)

print("Graph type:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Directed:", G.is_directed())

Graph type: MultiDiGraph
Nodes: 2569
Edges: 5850
Directed: True


In [21]:
from IPython.display import display

from routing.config import ORIGIN, DESTINATION
from routing.map_view import interactive_map

widget, state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    max_snap_distance_m=150
)

display(widget)

In [22]:
print("Origin:", state["origin_node"])
print("Destination:", state["destination_node"])
print("Last click:", state["last_click_latlon"])

assert state["origin_node"] in G
assert state["destination_node"] in G
assert state["last_click_latlon"] is not None

print("PASS: Interactive Map")

Origin: 25291778
Destination: 26389375
Last click: None


AssertionError: 

In [23]:
from routing.traffic_pipeline import (
    collect_traffic_snapshot,
    make_routing_graph
)

snapshot, summary = collect_traffic_snapshot(
    G,
    client=None,
    max_requests=0
)

print("API requests:", summary["requests_made"])
print("Total edges:", summary["total_directed_edges"])
print("Statuses:", summary["counts"])

API requests: 0
Total edges: 5850
Statuses: {'missing': 5850, 'red': 0, 'yellow': 0, 'clear': 0, 'closed': 0}


In [25]:
free_time_graph = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow"
)

measured_time_graph = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured"
)

distance_graph = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured"
)

print("Free flow:", free_time_graph.number_of_edges())
print("Measured:", measured_time_graph.number_of_edges())
print("Distance:", distance_graph.number_of_edges())

print("PASS: Routing graphs prepared")

Free flow: 5850
Measured: 5850
Distance: 5850
PASS: Routing graphs prepared


In [1]:
from IPython.display import display

from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.config import ORIGIN, DESTINATION

G = load_road_graph(radius_m=2000)

widget, state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    max_snap_distance_m=150
)

display(widget)

In [2]:
print("Origin:", state["origin_node"])
print("Destination:", state["destination_node"])
print("Last click:", state["last_click_latlon"])

assert state["origin_node"] in G
assert state["destination_node"] in G
assert state["last_click_latlon"] is not None

print("PASS: Interactive Map")

Origin: 26787883
Destination: 26389375
Last click: (51.486622628412164, -0.18608093261718753)
PASS: Interactive Map


In [4]:
from routing.traffic_pipeline import collect_traffic_snapshot

snapshot, summary = collect_traffic_snapshot(
    G,
    client=None,
    max_requests=0
)

In [5]:
traffic_widget, traffic_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot
)

display(traffic_widget)